PROMPT ENGINEERING

In [10]:
!pip install -q -U llama-index llama-index-llms-groq llama-index-llms-ollama

In [11]:
import os
from llama_index.core import PromptTemplate
from llama_index.llms.ollama import Ollama

In [12]:
llm = Ollama(
    model="gemma3:1b",
    temperature=0.0
)

In [13]:
template_str = (
    "You are an expert AI assistant.\n"
    "Use ONLY the use provided context to answer the user's question. "
    "If the context is insufficient or does not mention the answer, reply exactly: "
    "'Not enough information.'\n\n"
    "Context:\n{context_str}\n\n"
    "User Question: {query_str}\n\n"
    "Answering Rules:\n"
    "1) Be concise and precise (3–6 sentences, unless the question requires more).\n"
    "2) Use bullet points for lists.\n"
    "3) At the end, include a 'Sources:' section with short snippets or filenames from the context you used.\n\n"
    "Final Answer:"
)

template = PromptTemplate(template_str)
sample_context = (
    "Transformers use a self-attention mechanism that lets each token attend "
    "to every other token in the sequence. This enables modeling of long-range "
    "dependencies without recurrence. Positional encodings inject order "
    "information, and multi-head attention captures diverse relations.\n\n"
    "The encoder stacks layers of self-attention and feed-forward networks to "
    "build contextual representations. The decoder uses masked self-attention "
    "to maintain causality and cross-attention to consult encoder outputs."
)

sample_user_query = "How do Transformers handle long-range dependencies?"

In [14]:
filled_prompt = template.format(
    context_str=sample_context,
    query_str=sample_user_query
)
print(filled_prompt)

You are an expert AI assistant.
Use ONLY the use provided context to answer the user's question. If the context is insufficient or does not mention the answer, reply exactly: 'Not enough information.'

Context:
Transformers use a self-attention mechanism that lets each token attend to every other token in the sequence. This enables modeling of long-range dependencies without recurrence. Positional encodings inject order information, and multi-head attention captures diverse relations.

The encoder stacks layers of self-attention and feed-forward networks to build contextual representations. The decoder uses masked self-attention to maintain causality and cross-attention to consult encoder outputs.

User Question: How do Transformers handle long-range dependencies?

Answering Rules:
1) Be concise and precise (3–6 sentences, unless the question requires more).
2) Use bullet points for lists.
3) At the end, include a 'Sources:' section with short snippets or filenames from the context you

In [15]:
response = llm.complete(prompt=filled_prompt)

print(response.text)

Transformers address long-range dependencies through the self-attention mechanism. Each token in the sequence attends to every other token, allowing the model to capture relationships regardless of distance. Positional encodings provide order information, and multi-head attention enables diverse relations. The encoder stacks layers of self-attention and feed-forward networks to create contextual representations. The decoder utilizes masked self-attention to maintain causality and cross-attention to leverage the encoder's outputs.

Sources:
*   "Transformers use a self-attention mechanism that lets each token attend to every other token in the sequence. This enables modeling of long-range dependencies without recurrence."
*   "The encoder stacks layers of self-attention and feed-forward networks to build contextual representations."



In [16]:
from llama_index.core import PromptTemplate
from llama_index.llms.ollama import Ollama


def run_llm(context: str, query: str) -> str:
    # Initialize LLM (Ollama)
    llm = Ollama(model="gemma3:1b", temperature=0)

    # Define prompt template
    template_str = (
        "You are an expert AI assistant.\n"
        "Use ONLY use the provided context to answer the user's question. "
        "If the context is insufficient or does not mention the answer, reply exactly: "
        "'Not enough information.'\n\n"
        "Context:\n{context_str}\n\n"
        "User Question: {query_str}\n\n"
        "Answering Rules:\n"
        "1) Be concise and precise (3–6 sentences, unless the question requires more).\n"
        "2) Use bullet points for lists.\n"
        "3) At the end, include a 'Sources:' section with short snippets or filenames from the context you used.\n\n"
        "Final Answer:"
    )

    template = PromptTemplate(template_str)
    filled_prompt = template.format(context_str=context, query_str=query)

    # Get response
    response = llm.complete(prompt=filled_prompt)
    return response.text

In [17]:
sample_context = (
    "Transformers use a self-attention mechanism that lets each token attend "
    "to every other token in the sequence. This enables modeling of long-range "
    "dependencies without recurrence. Positional encodings inject order "
    "information, and multi-head attention captures diverse relations.\n\n"
    "The encoder stacks layers of self-attention and feed-forward networks to "
    "build contextual representations. The decoder uses masked self-attention "
    "to maintain causality and cross-attention to consult encoder outputs."
)

sample_query = "How do Transformers handle long-range dependencies?"

output = run_llm(sample_context, sample_query)

print(output)

Here's an answer to your question:

Transformers effectively handle long-range dependencies through the self-attention mechanism. Unlike recurrent models, they don't rely on sequential processing, allowing the model to directly assess relationships between any two tokens in the input sequence. This is achieved through self-attention, where each token attends to all other tokens, capturing dependencies regardless of distance. Positional encodings provide information about the order of tokens, and multi-head attention allows the model to consider diverse relations. The encoder stacks layers of these mechanisms to create contextual representations, and the decoder utilizes masked self-attention to ensure causality and cross-attention to leverage the encoder's output.

Sources:
*   [https://www.tensorflow.org/api_docs/python/transformers/models/TransformerEncoder](https://www.tensorflow.org/api_docs/python/transformers/models/TransformerEncoder)
*   [https://www.deeplearning.ai/blog/transf

FEW SHOT PROMPTING

In [1]:
!pip install -q -U llama-index llama-index-llms-groq

In [ ]:
# --- FEW-SHOT ---
import os
from typing import List, Dict
from llama_index.core import PromptTemplate
from llama_index.llms.groq import Groq

os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"

def run_llm_fewshot(context: str, query: str, examples: List[Dict[str, str]]) -> str:
    llm = Groq(model="llama-3.3-70b-versatile", temperature=0)

    examples_str = "\n".join(
        f"Example {i+1}:\nContext:\n{ex.get('context','')}\n"
        f"Question: {ex.get('question','')}\n"
        f"Answer: {ex.get('answer','')}\n"
        for i, ex in enumerate(examples)
    )

    template_str = (
        "You are an expert AI assistant.\n"
        "Use ONLY the provided context to answer the user's question. "
        "If the context is insufficient or does not mention the answer, reply exactly: "
        "'Not enough information.'\n\n"
        "Follow the style and reasoning illustrated by the examples.\n\n"
        "Examples:\n{examples}\n"
        "--- End of Examples ---\n\n"
        "Context:\n{context_str}\n\n"
        "User Question: {query_str}\n\n"
        "Answering Rules:\n"
        "1) Be concise and precise (3–6 sentences, unless the question requires more).\n"
        "2) Use bullet points for lists.\n"
        "3) At the end, include a 'Sources:' section with short snippets or filenames from the context you used.\n\n"
        "Final Answer:"
    )
    prompt = PromptTemplate(template_str).format(
        examples=examples_str, context_str=context, query_str=query
    )
    return llm.complete(prompt=prompt).text

In [5]:
# Few-Shot: Add examples so the model mimics your style
shots = [
    {
        "context": "Positional encodings inject order information into sequences.",
        "question": "Why are positional encodings needed?",
        "answer": (
            "They give the model a sense of word order.\n"
            "- Without them, the model treats tokens as a bag of words.\n"
            "- Encodings ensure the sequence structure is preserved.\n"
            "Sources: lecture_notes.txt"
        )
    },
    {
        "context": "Multi-head attention projects queries, keys, and values into multiple subspaces.",
        "question": "What is the benefit of multi-head attention?",
        "answer": (
            "It lets the model learn from different representation subspaces.\n"
            "- Captures diverse relationships.\n"
            "- Improves contextual understanding.\n"
            "Sources: attention_paper.pdf"
        )
    },
]

context_text = (
    "Context from attention_mechanism.pdf"
    "In the attention mechanism, softmax is used on the similarity scores "
    "between queries and keys to produce attention weights."
)

query_text = "What does softmax do in attention?"

ansk = run_llm_fewshot(context=context_text, query=query_text, examples=shots)

print(ansk)

In the attention mechanism, softmax is applied to the similarity scores between queries and keys. This process produces attention weights, which are used to compute the weighted sum of the values. The softmax function:
* Normalizes the similarity scores
* Ensures the attention weights add up to 1
* Allows the model to focus on the most relevant keys. 
The resulting attention weights determine how much attention is allocated to each key. 

Sources: attention_mechanism.pdf
